# 02 · Indexation du corpus : encodage, contrôles de précision et export pour le site

**Ticket #30.** Ce notebook transforme les passages produits par le notebook 01 en vecteurs, contrôle que ces
vecteurs sont ceux que le navigateur retrouvera, et exporte l'index du site ainsi que les index destinés à
l'évaluation.

| Entrée | Sortie |
|---|---|
| `data/decoupage/passages/` : tables de passages et manifeste du notebook 01 | `data/indexation/site/` : index du site (découpage M2), métadonnées et en-tête |
| modèles figés : `intfloat/multilingual-e5-small` (référence PyTorch) et `Xenova/multilingual-e5-small` (ONNX, servi au navigateur) | `data/indexation/variantes/` : un index par découpage, pour le notebook 03 |
| | `data/indexation/controle/` : requêtes et résultats du contrôle dans le navigateur |

**Ce que ce notebook établit** : l'intégrité des entrées, la parité entre le modèle de référence et le modèle
ONNX, l'effet de la quantification sur le classement des passages, mesuré en Python puis **dans le navigateur**,
le choix de la précision servie au site, et l'intégrité de l'export vérifiée par le navigateur lui-même.

**Ce qu'il n'établit pas** : la qualité de la recherche. Elle exige le jeu de questions annotées, en préparation,
et fera l'objet du notebook 03. Les requêtes utilisées ici servent à **comparer des précisions de calcul**, pas
à juger des résultats.

Le support `docs/etudes/indexation/01_encodeur_e5_small.pdf` explique la théorie utilisée ici : tokenisation,
moyenne des tokens, préfixes, quantification, lecture des scores.

**Sommaire**
1. Configuration et reproductibilité
2. Entrées : intégrité des passages du notebook 01
3. Modèles : référence PyTorch et modèles ONNX du navigateur
4. Contrôle 1 : parité entre le modèle ONNX et la référence
5. Requêtes de contrôle
6. Contrôle 2 : effet de la quantification q8 sur le classement
7. Encodage des découpages
8. Contrôle 3 : fidélité de l'encodeur dans le navigateur, et choix de la précision servie
9. Export de l'index du site
10. Export des index d'évaluation
11. Exemples de recherche
12. Contrôle 4 : intégrité de l'export, vérifiée dans le navigateur
13. Manifeste et synthèse

## 1. Configuration et reproductibilité

**Environnement.** Le notebook s'exécute dans le `.venv` de la racine du dépôt, construit depuis le fichier figé
de l'indexation, qui contient aussi toutes les dépendances du notebook 01 :

```
uv venv --seed --python 3.12 .venv
uv pip sync --python .venv/bin/python scripts/indexation/requirements-lock.txt
```

Dans VS Code : *Select Kernel*, *Python Environments*, `.venv`. Le premier lancement télécharge les deux fichiers
ONNX (448,5 et 112,8 Mio) et le modèle de référence (470 Mio) dans le cache Hugging Face, hors du dépôt.
Les sorties vont dans `data/indexation/`, que git ignore : la cellule le vérifie avant d'écrire.

In [1]:
import os
import warnings

# barres de progression des bibliothèques Hugging Face : sans intérêt ici, et bruyantes sans ipywidgets
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
warnings.filterwarnings("ignore", message="IProgress not found")

import gzip
import hashlib
import json
import platform
import subprocess
import sys
import time
from datetime import datetime, timezone
from math import comb
from pathlib import Path

import numpy as np
import onnxruntime as ort
import pandas as pd
import sentence_transformers
import tokenizers
import torch
from huggingface_hub import HfApi, hf_hub_download
from sentence_transformers import SentenceTransformer
from tokenizers import Tokenizer
from transformers.utils import logging as journal_transformers

journal_transformers.disable_progress_bar()

try:
    display
except NameError:  # exécution hors Jupyter : affichage texte
    def display(objet):
        """Affichage de repli quand le code tourne hors de Jupyter."""
        print(objet.to_string() if isinstance(objet, pd.DataFrame) else objet)

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 220)
GRAINE = 20260927
torch.manual_seed(GRAINE)


def racine_depot(depart: Path) -> Path:
    """Remonte les dossiers parents jusqu'à la racine du dépôt git, celle qui contient .git."""
    for dossier in (depart, *depart.parents):
        if (dossier / ".git").exists():
            return dossier
    raise FileNotFoundError(f"aucun dépôt git au-dessus de {depart}")


RACINE = racine_depot(Path.cwd().resolve())
ENTREES = RACINE / "data" / "decoupage" / "passages"
SORTIES = RACINE / "data" / "indexation"
SITE, VARIANTES, CONTROLE, CACHE = (SORTIES / d for d in ("site", "variantes", "controle", "cache"))
for dossier in (SITE, VARIANTES, CONTROLE, CACHE):
    dossier.mkdir(parents=True, exist_ok=True)
ignore = subprocess.run(["git", "-C", str(RACINE), "check-ignore", "-q", str(SORTIES)]).returncode == 0
assert ignore, "data/indexation n'est pas ignoré par git : corriger l'exclusion avant d'aller plus loin"

VERSIONS = {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
            "torch": torch.__version__, "sentence-transformers": sentence_transformers.__version__,
            "onnxruntime": ort.__version__, "tokenizers": tokenizers.__version__}
print("interpréteur :", sys.executable)
assert sys.version_info[:2] == (3, 12), f"Python {platform.python_version()} : le notebook est validé en Python 3.12"
if Path(sys.prefix).resolve() != (RACINE / ".venv").resolve():
    print("attention : l'interpréteur n'est pas le .venv du dépôt ; les versions peuvent différer du fichier figé")
print("racine du dépôt :", RACINE)
print(VERSIONS)

interpréteur : /Users/mahe/Desktop/M1_DTA/NLP_PROJET/data_tigers_nlp/.venv/bin/python
racine du dépôt : /Users/mahe/Desktop/M1_DTA/NLP_PROJET/data_tigers_nlp
{'python': '3.12.11', 'numpy': '2.5.3', 'pandas': '3.0.6', 'torch': '2.14.0', 'sentence-transformers': '6.1.0', 'onnxruntime': '1.30.0', 'tokenizers': '0.23.2'}


**Paramètres.** Tous réunis ici. Les deux préfixes sont ceux du modèle E5 : `passage: ` pour les passages,
`query: ` pour les questions. Le seuil de parité et les critères des contrôles navigateur sont fixés **avant**
toute mesure, pour qu'aucun seuil ne soit choisi après coup au vu des résultats.

In [2]:
PREFIXE_PASSAGE = "passage: "
PREFIXE_REQUETE = "query: "
BUDGET = 512                    # longueur maximale de l'encodeur, tokens spéciaux compris
DIMENSION = 384
TAILLE_LOT = 32
SEUIL_PARITE = 0.99999          # contrôle 1 : cosinus minimal entre ONNX fp32 et la référence
N_REQUETES_NAVIGATEUR = 20      # contrôle 4 : titres-questions tirés au hasard, en plus des 4 requêtes d'exemple
TOLERANCE_EGALITE = 1e-6        # contrôle 4 : deux scores plus proches que cela sont des ex aequo
PRECISIONS_8_BITS = {"q8": "onnx/model_quantized.onnx", "int8": "onnx/model_int8.onnx", "uint8": "onnx/model_uint8.onnx"}
MODELE_REF = {"depot": "intfloat/multilingual-e5-small", "revision": "614241f622f53c4eeff9890bdc4f31cfecc418b3"}
MODELE_ONNX = {"depot": "Xenova/multilingual-e5-small", "revision": "761b726dd34fb83930e26aab4e9ac3899aa1fa78",
               "fichiers": {"fp32": "onnx/model.onnx", "q8": "onnx/model_quantized.onnx"}}
# découpages à indexer : table du notebook 01 et colonne à encoder
DECOUPAGES = {
    "m2": ("m2", "entree_encodeur"),                    # méthode retenue, index du site
    "m2_sans_entete": ("m2", "entree_sans_entete"),     # mesure de l'effet de l'en-tête
    "m2_256": ("m2_256", "entree_encodeur"),            # mesure de l'effet du budget
    "m0": ("m0", "entree_encodeur"),                    # référence : découpage de l'éditeur
    "m1": ("m1", "entree_encodeur"),                    # fenêtre fixe
    "m3": ("m3", "entree_encodeur"),                    # découpage sémantique
}
DECOUPAGE_SITE = "m2"
REQUETES_EXEMPLE = [
    "Quelles mentions sont interdites sur le bulletin de paie ?",
    "Mon employeur peut-il me licencier pendant un arrêt maladie ?",
    "Combien de temps peut durer la période d'essai d'un CDI ?",
    "Peut-on être viré quand on est en arrêt de travail ?",
]

**Pour la soutenance.**

- Mêmes conventions que le notebook 01 : chemins depuis la racine, données hors de git, interpréteur affiché.
- Les seuils de décision sont écrits avant les mesures : on ne peut pas les ajuster pour que « ça passe ».
- Les deux modèles sont figés par leur révision : un autre commit du Hub donnerait d'autres vecteurs.

## 2. Entrées : intégrité des passages du notebook 01

Le notebook 02 ne recompose rien : il encode la colonne `entree_encodeur` telle que le notebook 01 l'a écrite.
Avant d'encoder, on vérifie que les fichiers sont bien ceux que décrit le manifeste du notebook 01 (empreinte du
fichier et empreinte du contenu), que chaque entrée porte le préfixe `passage: `, et que le nombre de tokens,
**recompté ici**, respecte la limite de 512.

In [3]:
def sha256_fichier(chemin: Path) -> str:
    """Empreinte SHA-256 d'un fichier, lue par blocs de 1 Mio."""
    h = hashlib.sha256()
    with open(chemin, "rb") as f:
        for bloc in iter(lambda: f.read(1 << 20), b""):
            h.update(bloc)
    return h.hexdigest()


def empreinte_contenu(ids, entrees) -> str:
    """Empreinte du contenu encodable, calculée exactement comme au notebook 01."""
    h = hashlib.sha256()
    for pid, entree in zip(ids, entrees):
        h.update(f"{pid}\x1f{entree}\x1e".encode())
    return h.hexdigest()


manifeste01 = json.loads((ENTREES / "manifeste.json").read_text(encoding="utf-8"))
TABLES, lignes = {}, []
for code in sorted({table for table, _ in DECOUPAGES.values()}):
    decrit = manifeste01["methodes"][code]
    chemin = RACINE / decrit["fichier"]
    assert sha256_fichier(chemin) == decrit["sha256_fichier"], f"{code} : fichier différent de celui du manifeste"
    P = pd.read_parquet(chemin)
    assert empreinte_contenu(P.passage_id, P.entree_encodeur) == decrit["empreinte_contenu"], f"{code} : contenu altéré"
    assert P.passage_id.is_unique and len(P) == decrit["passages"]
    TABLES[code] = P
    lignes.append({"table": code, "passages": len(P), "empreinte du fichier": "conforme", "empreinte du contenu": "conforme"})
display(pd.DataFrame(lignes))

tok = Tokenizer.from_pretrained(MODELE_ONNX["depot"], revision=MODELE_ONNX["revision"])
lignes = []
for nom, (table, colonne) in DECOUPAGES.items():
    textes = TABLES[table][colonne]
    assert textes.str.startswith(PREFIXE_PASSAGE).all(), f"{nom} : préfixe manquant"
    assert (textes.str.len() > len(PREFIXE_PASSAGE)).all(), f"{nom} : entrée vide"
    n = np.array([len(e.ids) for e in tok.encode_batch(textes.tolist())])
    if colonne == "entree_encodeur":
        assert (n == TABLES[table].n_tokens.to_numpy()).all(), f"{nom} : comptage différent de celui du notebook 01"
    assert n.max() <= BUDGET, f"{nom} : {n.max()} tokens, au-delà de la limite"
    lignes.append({"découpage": nom, "passages": len(textes), "tokens moyenne": round(n.mean(), 1), "max": int(n.max())})
display(pd.DataFrame(lignes))

,table,passages,empreinte du fichier,empreinte du contenu
0,m0,5284,conforme,conforme
1,m1,5317,conforme,conforme
2,m2,4240,conforme,conforme
3,m2_256,7009,conforme,conforme
4,m3,4229,conforme,conforme


,découpage,passages,tokens moyenne,max
0,m2,4240,259.1,512
1,m2_sans_entete,4240,228.5,501
2,m2_256,7009,174.1,256
3,m0,5284,253.1,440
4,m1,5317,258.7,329
5,m3,4229,260.8,493


**Pour la soutenance.**

- Le notebook 02 part de fichiers dont l'empreinte est vérifiée : on sait qu'il encode exactement ce que le 01 a produit.
- Le nombre de tokens est recompté indépendamment, et aucune entrée ne dépasse 512 : l'encodeur ne tronquera rien.

## 3. Modèles : référence PyTorch et modèles ONNX du navigateur

Deux familles de fichiers pour un même modèle :

- **la référence** : `intfloat/multilingual-e5-small` exécuté par PyTorch en fp32, via sentence-transformers,
  qui applique la moyenne des tokens puis la normalisation ;
- **les fichiers ONNX** de `Xenova/multilingual-e5-small`, ceux que Transformers.js charge dans le navigateur :
  `model.onnx` (fp32) et `model_quantized.onnx` (q8, chargé par défaut en WASM).

Les fichiers ONNX ne renvoient qu'un vecteur par token (`last_hidden_state`). La fonction `encoder_onnx`
reproduit donc ce que fait Transformers.js avec les options `{ pooling: "mean", normalize: true }` : moyenne
des vecteurs des tokens réels, pondérée par le masque d'attention, puis division par la norme. Le calcul est
fait en float32, comme dans le navigateur.

In [4]:
ref = SentenceTransformer(MODELE_REF["depot"], revision=MODELE_REF["revision"], device="cpu")
print(ref)

hub = HfApi().model_info(MODELE_ONNX["depot"], revision=MODELE_ONNX["revision"], files_metadata=True)
empreintes_hub = {s.rfilename: s.lfs.sha256 for s in hub.siblings if s.lfs}
SESSIONS, FICHIERS_ONNX = {}, {}
for precision, fichier in MODELE_ONNX["fichiers"].items():
    chemin = Path(hf_hub_download(MODELE_ONNX["depot"], fichier, revision=MODELE_ONNX["revision"]))
    empreinte = sha256_fichier(chemin)
    assert empreinte == empreintes_hub[fichier], f"{fichier} : empreinte différente de celle du Hub"
    SESSIONS[precision] = ort.InferenceSession(str(chemin), providers=["CPUExecutionProvider"])
    FICHIERS_ONNX[precision] = {"fichier": fichier, "sha256": empreinte, "taille_Mio": round(chemin.stat().st_size / 2**20, 1)}
    entrees = [i.name for i in SESSIONS[precision].get_inputs()]
    sorties = [o.name for o in SESSIONS[precision].get_outputs()]
    print(f"{precision:5} {fichier:28} {FICHIERS_ONNX[precision]['taille_Mio']:6.1f} Mio  entrées {entrees}  sorties {sorties}")
assert entrees == ["input_ids", "attention_mask", "token_type_ids"] and sorties == ["last_hidden_state"]
ID_REMPLISSAGE = tok.token_to_id("<pad>")
print("identifiant du token de remplissage :", ID_REMPLISSAGE)

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({'module_input_name': 'sentence_embedding', 'module_output_name': 'sentence_embedding'})
)


fp32  onnx/model.onnx               448.5 Mio  entrées ['input_ids', 'attention_mask', 'token_type_ids']  sorties ['last_hidden_state']
q8    onnx/model_quantized.onnx     112.8 Mio  entrées ['input_ids', 'attention_mask', 'token_type_ids']  sorties ['last_hidden_state']
identifiant du token de remplissage : 1


In [5]:
def encoder_ref(textes: list) -> np.ndarray:
    """Encodage de référence : PyTorch fp32, moyenne des tokens et normalisation par sentence-transformers."""
    return ref.encode(list(textes), batch_size=TAILLE_LOT, normalize_embeddings=True,
                      convert_to_numpy=True, show_progress_bar=False).astype(np.float32)


def encoder_onnx(textes: list, precision: str) -> np.ndarray:
    """Encodage par un fichier ONNX, comme Transformers.js : tokenisation, moyenne masquée, normalisation L2."""
    session = SESSIONS[precision]
    encs = tok.encode_batch(list(textes))
    longueurs = np.array([len(e.ids) for e in encs])
    assert longueurs.max() <= BUDGET, "une entrée dépasse la limite : elle serait tronquée"
    ordre = np.argsort(longueurs, kind="stable")      # lots de longueurs voisines : moins de remplissage
    sortie = np.empty((len(encs), DIMENSION), dtype=np.float32)
    for debut in range(0, len(ordre), TAILLE_LOT):
        lot = ordre[debut:debut + TAILLE_LOT]
        L = int(longueurs[lot].max())
        ids = np.full((len(lot), L), ID_REMPLISSAGE, dtype=np.int64)
        masque = np.zeros((len(lot), L), dtype=np.int64)
        for k, i in enumerate(lot):
            ids[k, :longueurs[i]] = encs[i].ids
            masque[k, :longueurs[i]] = 1
        etats = session.run(None, {"input_ids": ids, "attention_mask": masque,
                                   "token_type_ids": np.zeros_like(ids)})[0]            # (lot, L, 384)
        m = masque[..., None].astype(np.float32)
        moyenne = (etats * m).sum(axis=1) / m.sum(axis=1)                                # moyenne des tokens réels
        sortie[lot] = moyenne / np.linalg.norm(moyenne, axis=1, keepdims=True)          # normalisation L2
    return sortie


def encoder(textes: list, moteur: str, nom: str) -> np.ndarray:
    """Encode avec le moteur demandé (ref, fp32 ou q8), en réutilisant un cache dont la clé dépend de tout ce qui fixe le résultat."""
    textes = list(textes)
    revision = MODELE_REF["revision"] if moteur == "ref" else FICHIERS_ONNX[moteur]["sha256"]
    # textes, moteur, révision ou fichier du modèle, taille de lot (le remplissage d'un lot change les derniers
    # chiffres) et versions des bibliothèques : un changement de l'un d'eux invalide le cache
    contexte = f"{moteur}|{revision}|{TAILLE_LOT}|{json.dumps(VERSIONS, sort_keys=True)}"
    cle = hashlib.sha256(("\x1e".join(textes) + "\x1d" + contexte).encode()).hexdigest()[:16]
    fichier = CACHE / f"{nom}_{moteur}_{cle}.npy"
    duree = fichier.with_suffix(".json")      # durée du calcul d'origine, conservée pour le manifeste
    if fichier.exists() and duree.exists():
        DUREES[(nom, moteur)] = json.loads(duree.read_text())["secondes"]
        return np.load(fichier)
    debut = time.perf_counter()
    E = encoder_ref(textes) if moteur == "ref" else encoder_onnx(textes, moteur)
    DUREES[(nom, moteur)] = time.perf_counter() - debut
    np.save(fichier, E)
    duree.write_text(json.dumps({"secondes": DUREES[(nom, moteur)], "textes": len(textes)}))
    return E


DUREES = {}

**Pour la soutenance.**

- Le navigateur n'exécute pas PyTorch mais des fichiers ONNX : il faut vérifier que les deux calculent la même chose.
- Les fichiers ONNX sont contrôlés par l'empreinte publiée sur le Hub.
- `encoder_onnx` reproduit ligne à ligne la moyenne masquée et la normalisation que Transformers.js applique.

## 4. Contrôle 1 : parité entre le modèle ONNX et la référence

**Question.** Le fichier `model.onnx` (fp32), exécuté par notre fonction `encoder_onnx`, donne-t-il les mêmes
vecteurs que la référence PyTorch ? Si oui, notre reproduction du calcul du navigateur est juste, et toute
différence observée ensuite avec le q8 vient de la quantification seule.

**Critères fixés à l'avance.** Tokenisation identique pour 100 % des entrées ; cosinus minimal d'au moins
0,99999 entre les deux encodages de chaque passage M2.

In [6]:
passages_site = TABLES["m2"].entree_encodeur.tolist()
ecarts_tok = [t for t in passages_site if ref.tokenizer(t)["input_ids"] != tok.encode(t).ids]
print(f"tokenisation : {len(passages_site) - len(ecarts_tok)} entrées identiques sur {len(passages_site)}")
assert not ecarts_tok

E_ref = encoder(passages_site, "ref", "m2")
E_fp32 = encoder(passages_site, "fp32", "m2")
cos_parite = np.einsum("ij,ij->i", E_ref, E_fp32)
print(f"cosinus ONNX fp32 et référence : minimum {cos_parite.min():.8f}, médiane {np.median(cos_parite):.8f} ; "
      f"écart absolu maximal sur une composante {np.abs(E_ref - E_fp32).max():.2e}")
assert cos_parite.min() >= SEUIL_PARITE, "parité non atteinte : la reproduction du calcul du navigateur est à revoir"

tokenisation : 4240 entrées identiques sur 4240
cosinus ONNX fp32 et référence : minimum 0.99999976, médiane 1.00000000 ; écart absolu maximal sur une composante 2.76e-07


**Pour la soutenance.**

- Deux implémentations indépendantes du même modèle donnent les mêmes vecteurs, aux arrondis de calcul près.
- La moyenne masquée et la normalisation de `encoder_onnx` sont donc justes ; le reste du notebook peut s'y fier.

## 5. Requêtes de contrôle

Le jeu de questions annotées n'existe pas encore. Pour comparer des **précisions de calcul**, il suffit de
requêtes réalistes : on prend les titres de section formulés en questions (« Quelle est la durée de la période
d'essai ? »), nombreux dans les fiches, préfixés par `query: `.

**Limite à ne pas oublier.** Ces titres figurent dans l'en-tête des passages M2 : la recherche les retrouve
trop facilement. Ils conviennent pour vérifier que deux calculs classent les passages de la même façon ; ils ne
disent **rien** de la qualité de la recherche, et ne servent jamais à comparer les découpages entre eux.

In [7]:
TITRES_QUESTIONS = sorted({t.strip() for t in TABLES["m2"].section_titre if t.strip().endswith("?")})
REQUETES = [PREFIXE_REQUETE + t for t in TITRES_QUESTIONS]
print(f"{len(TITRES_QUESTIONS)} titres-questions distincts sur {TABLES['m2'].section_titre.nunique()} titres de section")
print("exemples :", TITRES_QUESTIONS[:3])
assert all(ref.tokenizer(q)["input_ids"] == tok.encode(q).ids for q in REQUETES)

# liste lue par la page de fidélité du navigateur (section 8)
(CONTROLE / "requetes_fidelite.json").write_text(
    json.dumps({"prefixe": PREFIXE_REQUETE, "requetes": TITRES_QUESTIONS}, ensure_ascii=False), encoding="utf-8")

Q_ref = encoder(REQUETES, "ref", "requetes")
Q_fp32 = encoder(REQUETES, "fp32", "requetes")
Q_q8 = encoder(REQUETES, "q8", "requetes")
cos_q = np.einsum("ij,ij->i", Q_ref, Q_fp32)
print(f"parité sur les requêtes : cosinus minimal {cos_q.min():.8f}")
assert cos_q.min() >= SEUIL_PARITE

822 titres-questions distincts sur 1301 titres de section
exemples : ['A quel moment doit intervenir l’examen médical ?', 'A quel âge peut-on travailler en France ?', 'A quelles exigences une commande bimanuelle doit-elle répondre ?']
parité sur les requêtes : cosinus minimal 0.99999976


**Pour la soutenance.**

- En attendant le jeu annoté, les titres-questions donnent des centaines de requêtes réalistes, sans coût humain.
- Ils ne mesurent que l'accord entre deux calculs, jamais la qualité : leur texte figure dans l'en-tête des passages.

## 6. Contrôle 2 : effet de la quantification q8 sur le classement

Dans le navigateur, la question sera encodée par le modèle q8. Deux façons de construire l'index sont possibles :

| Configuration | Index des passages | Requête | Rôle |
|---|---|---|---|
| référence | PyTorch fp32 | PyTorch fp32 | le classement sans aucune quantification |
| A | PyTorch fp32 | ONNX q8 | index calculé une fois au mieux, requête du navigateur |
| B | ONNX q8 | ONNX q8 | index et requête calculés par le même modèle quantifié |

Une troisième configuration, index ONNX fp32 et requête q8, est inutile : le contrôle 1 a montré que l'index
ONNX fp32 est identique à l'index PyTorch.

**Mesures.** Pour chaque configuration, comparée à la référence : part des requêtes dont le premier résultat est
le même, recouvrement moyen des 5 et des 10 premiers résultats, part des requêtes dont le premier résultat de
référence reste parmi les 5 premiers (ce que voit un visiteur à qui l'on affiche 5 résultats), et rang le plus
bas auquel il tombe.

**Règle de décision, fixée avant la mesure.** On retient A, la plus simple et indépendante de la quantification,
sauf si B reproduit significativement mieux le premier résultat de référence (test exact de McNemar sur les
requêtes discordantes, au seuil de 5 %).

In [8]:
E_q8 = encoder(passages_site, "q8", "m2")
cos_quant = np.einsum("ij,ij->i", E_ref, E_q8)
print(f"passages, cosinus q8 et référence : minimum {cos_quant.min():.5f}, médiane {np.median(cos_quant):.5f}")
print(f"requêtes, cosinus q8 et référence : minimum {np.einsum('ij,ij->i', Q_ref, Q_q8).min():.5f}, "
      f"médiane {np.median(np.einsum('ij,ij->i', Q_ref, Q_q8)):.5f}")


def classement(Q: np.ndarray, E: np.ndarray, k: int = 10) -> tuple:
    """Scores complets et indices des k meilleurs passages pour chaque requête, par produit scalaire."""
    S = Q @ E.T
    return S, np.argsort(-S, axis=1, kind="stable")[:, :k]


def accord(top: np.ndarray, S: np.ndarray, top_ref: np.ndarray) -> dict:
    """Accord d'un classement avec celui de référence : premier résultat, recouvrements, rang du premier de référence."""
    rang_premier = np.array([(S[i] > S[i, top_ref[i, 0]]).sum() + 1 for i in range(len(S))])
    return {"même premier résultat %": round(100 * np.mean(top[:, 0] == top_ref[:, 0]), 1),
            "recouvrement du top 5 %": round(100 * np.mean([len(set(a[:5]) & set(b[:5])) / 5 for a, b in zip(top, top_ref)]), 1),
            "recouvrement du top 10 %": round(100 * np.mean([len(set(a) & set(b)) / 10 for a, b in zip(top, top_ref)]), 1),
            "premier de référence dans le top 5 %": round(100 * np.mean(rang_premier <= 5), 1),
            "rang du premier de référence, maximum": int(rang_premier.max())}


def mcnemar_exact(a_juste: np.ndarray, b_juste: np.ndarray) -> dict:
    """Test exact de McNemar, bilatéral : ne compte que les requêtes où une seule des deux configurations est juste."""
    b = int(np.sum(a_juste & ~b_juste))
    c = int(np.sum(~a_juste & b_juste))
    n = b + c
    p = 1.0 if n == 0 else min(1.0, 2 * sum(comb(n, i) for i in range(min(b, c) + 1)) / 2 ** n)
    return {"A juste seule": b, "B juste seule": c, "p": p}


S_ref, top_ref = classement(Q_ref, E_ref)
S_A, top_A = classement(Q_q8, E_ref)
S_B, top_B = classement(Q_q8, E_q8)
tableau_quant = pd.DataFrame({"A : index fp32, requête q8": accord(top_A, S_A, top_ref),
                              "B : index q8, requête q8": accord(top_B, S_B, top_ref)}).T
display(tableau_quant)
test = mcnemar_exact(top_A[:, 0] == top_ref[:, 0], top_B[:, 0] == top_ref[:, 0])
print("test exact de McNemar sur le premier résultat :", test)
CONFIGURATION = "B" if (test["p"] < 0.05 and test["B juste seule"] > test["A juste seule"]) else "A"
print("configuration retenue par la règle fixée à l'avance :", CONFIGURATION)

passages, cosinus q8 et référence : minimum 0.99817, médiane 0.99891
requêtes, cosinus q8 et référence : minimum 0.99825, médiane 0.99898


,même premier résultat %,recouvrement du top 5 %,recouvrement du top 10 %,premier de référence dans le top 5 %,"rang du premier de référence, maximum"
"A : index fp32, requête q8",96.5,96.0,95.7,100.0,2.0
"B : index q8, requête q8",93.2,94.5,94.0,100.0,4.0


test exact de McNemar sur le premier résultat : {'A juste seule': 38, 'B juste seule': 11, 'p': 0.00014197068519905542}
configuration retenue par la règle fixée à l'avance : A


**Stockage de l'index en entiers de 8 bits.** L'index du site pourrait être livré en int8 (une échelle par
dimension) au lieu de fp32, soit quatre fois moins lourd (tailles affichées ci-dessous). On mesure ce que coûte
cette compression au classement, dans la configuration retenue.

In [9]:
E_config = E_ref if CONFIGURATION == "A" else E_q8
echelle = np.abs(E_config).max(axis=0) / 127.0
E_int8 = np.clip(np.round(E_config / echelle), -127, 127).astype(np.int8)
S_cfg, top_cfg = classement(Q_q8, E_config)
S_i8, top_i8 = classement(Q_q8, E_int8.astype(np.float32) * echelle)
display(pd.DataFrame({"index int8 contre index fp32 (même requête q8)": accord(top_i8, S_i8, top_cfg)}).T)
print(f"taille : fp32 {E_config.nbytes / 2**20:.2f} Mio, int8 {E_int8.nbytes / 2**20:.2f} Mio ; "
      f"le modèle q8 téléchargé par le navigateur pèse {FICHIERS_ONNX['q8']['taille_Mio']} Mio")

,même premier résultat %,recouvrement du top 5 %,recouvrement du top 10 %,premier de référence dans le top 5 %,"rang du premier de référence, maximum"
index int8 contre index fp32 (même requête q8),98.7,98.2,98.2,100.0,3.0


taille : fp32 6.21 Mio, int8 1.55 Mio ; le modèle q8 téléchargé par le navigateur pèse 112.8 Mio


**Limite découverte à l'exécution.** Ce contrôle simule le q8 avec ONNX Runtime en Python. La section 8 montre
que le navigateur exécute le même fichier q8 de façon moins fidèle : la simulation Python **sous-estime** l'erreur
réelle du site. La mesure qui fait foi pour le site est celle de la section 8, faite dans le navigateur.

**Pour la soutenance.**

- La quantification q8 ne change presque pas les vecteurs, mais le classement fin est sensible : on mesure donc
  l'effet sur les premiers résultats, pas seulement sur les vecteurs.
- La configuration est choisie par une règle écrite avant la mesure, avec un test apparié.
- L'index est livré en fp32 : l'int8 ferait gagner quelques Mio, négligeables face au modèle, au prix de premiers
  résultats modifiés.

## 7. Encodage des découpages

Tous les découpages sont encodés avec le moteur de la configuration retenue pour les passages : PyTorch fp32
pour A, ONNX q8 pour B. Contrôles sur chaque matrice : dimension, valeurs finies, norme égale à 1, et deux
entrées identiques doivent donner deux vecteurs identiques. Enfin, un échantillon de passages est réencodé un
par un : le résultat ne doit pas dépendre du lot dans lequel un passage a été encodé.

In [10]:
MOTEUR_INDEX = "ref" if CONFIGURATION == "A" else "q8"
INDEX, lignes = {}, []
for nom, (table, colonne) in DECOUPAGES.items():
    textes = TABLES[table][colonne].tolist()
    E = E_config if nom == DECOUPAGE_SITE else encoder(textes, MOTEUR_INDEX, nom)
    normes = np.linalg.norm(E, axis=1)
    assert E.shape == (len(textes), DIMENSION) and E.dtype == np.float32 and np.isfinite(E).all(), nom
    assert np.abs(normes - 1).max() < 1e-5, f"{nom} : norme hors de 1 ± 1e-5"
    doublons = pd.Series(range(len(textes))).groupby(pd.Series(textes)).apply(list)
    doublons = [ix for ix in doublons if len(ix) > 1]
    ecart_doublons = max((np.abs(E[ix] - E[ix[0]]).max() for ix in doublons), default=0.0)
    assert ecart_doublons < 1e-6, f"{nom} : deux entrées identiques ont des vecteurs différents"
    INDEX[nom] = E
    lignes.append({"découpage": nom, "passages": len(textes), "écart de norme maximal": f"{np.abs(normes - 1).max():.1e}",
                   "entrées en double": sum(len(ix) - 1 for ix in doublons), "taille fp32 Mio": round(E.nbytes / 2**20, 2)})
display(pd.DataFrame(lignes))

rng = np.random.default_rng(GRAINE)
echantillon = rng.choice(len(passages_site), 48, replace=False)
un_par_un = np.stack([(encoder_ref([passages_site[i]]) if MOTEUR_INDEX == "ref" else encoder_onnx([passages_site[i]], "q8"))[0]
                      for i in echantillon])
print(f"passage encodé seul ou dans un lot : écart maximal {np.abs(un_par_un - INDEX[DECOUPAGE_SITE][echantillon]).max():.2e}")
assert np.abs(un_par_un - INDEX[DECOUPAGE_SITE][echantillon]).max() < 1e-5

,découpage,passages,écart de norme maximal,entrées en double,taille fp32 Mio
0,m2,4240,1.8e-07,3,6.21
1,m2_sans_entete,4240,1.8e-07,57,6.21
2,m2_256,7009,1.8e-07,6,10.27
3,m0,5284,1.8e-07,0,7.74
4,m1,5317,1.8e-07,0,7.79
5,m3,4229,1.8e-07,2,6.19


passage encodé seul ou dans un lot : écart maximal 8.57e-08


**Pour la soutenance.**

- Six index, un par découpage, tous contrôlés : dimension 384, normes à 1, aucune valeur invalide.
- Deux passages au texte identique ont le même vecteur ; un passage a le même vecteur seul ou dans un lot.

## 8. Contrôle 3 : fidélité de l'encodeur dans le navigateur, et choix de la précision servie

**Pourquoi ce contrôle existe.** La première version de ce notebook fixait un critère unique au navigateur : un
cosinus d'au moins 0,9999 entre le vecteur calculé par le navigateur et celui du q8 calculé en Python. À la
première exécution, ce critère a **échoué** (cosinus minimal de 0,9942). Le diagnostic, fait dans le navigateur :

- en fp32, le navigateur donne exactement les vecteurs de la référence (cosinus minimal de 1,000000) ;
- la tokenisation de Transformers.js est identique à la nôtre, requête par requête ;
- une moyenne des tokens recalculée à la main redonne exactement le vecteur du navigateur.

Tokenisation, moyenne et normalisation sont donc justes. C'est l'**exécution du fichier q8 par ONNX Runtime Web**
qui diffère de celle d'ONNX Runtime en Python. Le critère reposait sur une hypothèse fausse, que les deux moteurs
calculent pareil. On le remplace par deux contrôles distincts : la fidélité de l'encodeur, mesurée ici **dans le
navigateur**, et l'intégrité de l'export (section 12).

**Procédure.** La page `scripts/indexation/fidelite_navigateur.html`, servie en local, encode les 822 requêtes de
la section 5 dans le navigateur avec chaque précision du modèle (q8, int8, uint8, fp16, fp32). Ses vecteurs sont
enregistrés dans `controle/fidelite_navigateur.json` et comparés ici à la référence PyTorch fp32, avec l'index fp32.

**Règle de choix, fixée avant la mesure.** Parmi les variantes 8 bits (q8, int8, uint8, 112,6 à 112,8 Mio), on
retient celle qui reproduit le plus souvent le premier résultat de référence ; q8, le choix par défaut de
Transformers.js, est gardé si l'écart n'est pas significatif (McNemar, 5 %). fp16 et fp32 sont mesurés pour situer
la perte, pas pour être servis : ils doublent ou quadruplent le téléchargement du premier chargement (224 et 449 Mio,
contre 113 Mio), à l'opposé de la cible d'environ 50 Mio que le rapport J2 fixe à la première visite (risque R2).

**Le diagnostic, rejoué et vérifié.** La page `scripts/indexation/diagnostic_navigateur.html` isole, sur les 24
requêtes du contrôle 4, les trois étapes de l'encodage dans le navigateur. La cellule suivante vérifie ses trois
conclusions : tokenisation identique à la nôtre, navigateur fp32 identique à la référence, moyenne des tokens
recalculée à la main identique au pipeline q8 du navigateur.

In [11]:
fichier_diag, DIAGNOSTIC = CONTROLE / "diagnostic_navigateur.json", None
fichier_ctrl = CONTROLE / "resultats_navigateur.json"
if fichier_diag.exists() and fichier_ctrl.exists():
    diag = json.loads(fichier_diag.read_text(encoding="utf-8"))
    ctrl = json.loads(fichier_ctrl.read_text(encoding="utf-8"))
    assert diag["questions"] == [r["question"] for r in ctrl["resultats"]], "diagnostic et contrôle sur des requêtes différentes"
    textes_diag = [PREFIXE_REQUETE + q for q in diag["questions"]]
    Q_diag_ref = encoder(textes_diag, "ref", "diagnostic")
    Q_diag_q8 = encoder(textes_diag, "q8", "diagnostic")
    F = np.array(diag["vecteurs_fp32"], dtype=np.float32)
    M = np.array(diag["vecteurs_q8_manuels"], dtype=np.float32)
    P8 = np.array([r["vecteur"] for r in ctrl["resultats"]], dtype=np.float32)

    def cos_lignes(A, B):
        """Cosinus ligne à ligne entre deux matrices de vecteurs."""
        return np.einsum("ij,ij->i", A, B) / np.linalg.norm(A, axis=1) / np.linalg.norm(B, axis=1)

    DIAGNOSTIC = {
        "tokenisation identique à la nôtre": f"{sum(t == tok.encode(x).ids for t, x in zip(diag['tokens'], textes_diag))} / {len(textes_diag)}",
        "navigateur fp32 contre référence, cosinus minimal": round(float(cos_lignes(F, Q_diag_ref).min()), 7),
        "moyenne manuelle contre pipeline q8 du navigateur, cosinus minimal": round(float(cos_lignes(M, P8).min()), 7),
        "navigateur q8 contre q8 de Python, cosinus minimal": round(float(cos_lignes(P8, Q_diag_q8).min()), 5),
        "q8 de Python contre référence, cosinus minimal": round(float(cos_lignes(Q_diag_q8, Q_diag_ref).min()), 5),
    }
    display(pd.Series(DIAGNOSTIC).to_frame("diagnostic"))
    assert all(t == tok.encode(x).ids for t, x in zip(diag["tokens"], textes_diag))
    assert cos_lignes(F, Q_diag_ref).min() >= SEUIL_PARITE and cos_lignes(M, P8).min() >= SEUIL_PARITE
else:
    print("diagnostic non encore effectué : ouvrir la page de diagnostic, puis réexécuter le notebook")

,diagnostic
tokenisation identique à la nôtre,24 / 24
"navigateur fp32 contre référence, cosinus minimal",1.0
"moyenne manuelle contre pipeline q8 du navigateur, cosinus minimal",1.0
"navigateur q8 contre q8 de Python, cosinus minimal",0.99419
"q8 de Python contre référence, cosinus minimal",0.99866


In [12]:
fichier_fidelite = CONTROLE / "fidelite_navigateur.json"
DTYPE_NAVIGATEUR, FIDELITE = "q8", None
if fichier_fidelite.exists():
    fid = json.loads(fichier_fidelite.read_text(encoding="utf-8"))
    assert fid["requetes"] == TITRES_QUESTIONS and fid["prefixe"] == PREFIXE_REQUETE, "la page a traité d'autres requêtes"
    assert fid["revision"] == MODELE_ONNX["revision"]
    lignes, justes = [], {}
    for dtype, res in fid["resultats"].items():
        V = np.array(res["vecteurs"], dtype=np.float32)
        cos_v = np.einsum("ij,ij->i", V, Q_ref)
        S_v, top_v = classement(V, E_ref)
        justes[dtype] = top_v[:, 0] == top_ref[:, 0]
        lignes.append({"précision": dtype, "cosinus avec la référence, minimum": round(float(cos_v.min()), 5),
                       "médiane": round(float(np.median(cos_v)), 5), **accord(top_v, S_v, top_ref),
                       "encodage ms par requête": round(res["encodage_ms"] / len(TITRES_QUESTIONS), 1)})
    FIDELITE = pd.DataFrame(lignes).set_index("précision")
    display(FIDELITE)
    # règle de choix fixée à l'avance
    meilleure = max(PRECISIONS_8_BITS, key=lambda d: justes[d].sum())
    test_dtype = mcnemar_exact(justes[meilleure], justes["q8"]) if meilleure != "q8" else None
    if meilleure != "q8" and test_dtype["p"] < 0.05 and test_dtype["A juste seule"] > test_dtype["B juste seule"]:
        DTYPE_NAVIGATEUR = meilleure
    if meilleure == "q8":
        print("meilleure variante 8 bits : q8, le choix par défaut de Transformers.js ; aucun test nécessaire")
    else:
        print(f"meilleure variante 8 bits : {meilleure} ; test exact de McNemar contre q8 : {test_dtype}")
    # la configuration A reste-t-elle la bonne avec les vrais vecteurs du navigateur ?
    V_nav = np.array(fid["resultats"][DTYPE_NAVIGATEUR]["vecteurs"], dtype=np.float32)
    S_Anav, top_Anav = classement(V_nav, E_ref)
    S_Bnav, top_Bnav = classement(V_nav, E_q8)
    test_ab = mcnemar_exact(top_Anav[:, 0] == top_ref[:, 0], top_Bnav[:, 0] == top_ref[:, 0])
    print(f"avec les vecteurs du navigateur ({DTYPE_NAVIGATEUR}) : A (index fp32) {100 * np.mean(top_Anav[:, 0] == top_ref[:, 0]):.1f} %, "
          f"B (index q8) {100 * np.mean(top_Bnav[:, 0] == top_ref[:, 0]):.1f} % de premier résultat identique ; McNemar {test_ab}")
else:
    print("mesure de fidélité non encore effectuée : ouvrir la page de fidélité, puis réexécuter le notebook ; précision par défaut q8")
print("précision servie au navigateur :", DTYPE_NAVIGATEUR)

,"cosinus avec la référence, minimum",médiane,même premier résultat %,recouvrement du top 5 %,recouvrement du top 10 %,premier de référence dans le top 5 %,"rang du premier de référence, maximum",encodage ms par requête
précision,,,,,,,,
q8,0.99196,0.99655,91.8,90.9,91.3,99.9,9,18.6
int8,0.97326,0.98956,85.8,82.9,83.5,99.3,23,18.1
uint8,0.98262,0.99445,90.8,88.6,89.2,99.9,8,18.2
fp16,1.00000,1.00000,99.9,100.0,100.0,100.0,2,16.7
fp32,1.00000,1.00000,100.0,100.0,100.0,100.0,1,16.6


meilleure variante 8 bits : q8, le choix par défaut de Transformers.js ; aucun test nécessaire


avec les vecteurs du navigateur (q8) : A (index fp32) 91.8 %, B (index q8) 91.4 % de premier résultat identique ; McNemar {'A juste seule': 19, 'B juste seule': 15, 'p': 0.6075913612730801}
précision servie au navigateur : q8


**Pour la soutenance.**

- Un critère fixé à l'avance a échoué : on l'a gardé visible, on a cherché la cause, et on a corrigé la méthode.
- Le navigateur en fp32 reproduit exactement la référence ; l'écart vient de l'exécution du q8 par ONNX Runtime Web.
- La précision servie est choisie sur des mesures faites dans le navigateur, par une règle écrite avant la mesure.

## 9. Export de l'index du site

Trois fichiers, à copier dans le dossier `public/` du site :

| Fichier | Contenu |
|---|---|
| `index_m2.f32` | la matrice des vecteurs, en float32 petit-boutiste, ligne par ligne : `nb_vecteurs × 384` nombres |
| `passages_m2.json` | un objet par passage, **dans le même ordre que les lignes de la matrice** : identifiant, fiche, section, chemin, liens, date, texte, articles |
| `index_m2.json` | l'en-tête : modèle, révision et précision à charger, préfixe et options de la requête, dimensions, empreintes des deux autres fichiers |

Le site doit lire l'en-tête d'abord et refuser l'index si le nombre de vecteurs, la dimension ou une empreinte ne
correspond pas : c'est la garantie qu'un vecteur ne sera jamais associé au mauvais passage. Côté navigateur,
`new Float32Array(await reponse.arrayBuffer())` relit la matrice sans conversion.

In [13]:
P_site = TABLES["m2"]
E_site = INDEX[DECOUPAGE_SITE]
chemin_bin, chemin_meta, chemin_entete = SITE / "index_m2.f32", SITE / "passages_m2.json", SITE / "index_m2.json"
E_site.astype("<f4").tofile(chemin_bin)
meta = [{"passage_id": r.passage_id, "titre_fiche": r.titre_fiche, "section_titre": r.section_titre,
         "chemin": list(r.chemin), "url": r.url, "url_citation": r.url_citation, "date_maj": r.date_maj,
         "texte": r.texte, "articles": list(r.articles)} for r in P_site.itertuples()]
chemin_meta.write_text(json.dumps(meta, ensure_ascii=False, separators=(",", ":")), encoding="utf-8")
fichier_servi = PRECISIONS_8_BITS[DTYPE_NAVIGATEUR]
entete_index = {
    "format": "index-e5-v1",
    "modele": {"depot": MODELE_ONNX["depot"], "revision": MODELE_ONNX["revision"], "dtype": DTYPE_NAVIGATEUR,
               "fichier": fichier_servi, "sha256": empreintes_hub[fichier_servi]},
    "requete": {"prefixe": PREFIXE_REQUETE, "pooling": "mean", "normalize": True},
    "index": {"fichier": chemin_bin.name, "type": "float32", "ordre_octets": "little-endian", "disposition": "ligne par ligne",
              "nb_vecteurs": int(E_site.shape[0]), "dimension": DIMENSION, "sha256": sha256_fichier(chemin_bin),
              "moteur": "PyTorch fp32 (intfloat/multilingual-e5-small)" if MOTEUR_INDEX == "ref" else "ONNX q8"},
    "metadonnees": {"fichier": chemin_meta.name, "nb_passages": len(meta), "sha256": sha256_fichier(chemin_meta)},
    "source": {"decoupage": "m2", "empreinte_contenu_notebook_01": manifeste01["methodes"]["m2"]["empreinte_contenu"]},
    "produit_le": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}
chemin_entete.write_text(json.dumps(entete_index, ensure_ascii=False, indent=2), encoding="utf-8")

# relecture : ce que le site lira doit redonner exactement ce qui est en mémoire
relu = np.fromfile(chemin_bin, dtype="<f4").reshape(-1, DIMENSION)
assert np.array_equal(relu, E_site)
meta_relue = json.loads(chemin_meta.read_text(encoding="utf-8"))
assert [m["passage_id"] for m in meta_relue] == P_site.passage_id.tolist()
entete_relu = json.loads(chemin_entete.read_text(encoding="utf-8"))
assert entete_relu["index"]["nb_vecteurs"] == len(meta_relue) == relu.shape[0]
assert entete_relu["index"]["sha256"] == sha256_fichier(chemin_bin) and entete_relu["metadonnees"]["sha256"] == sha256_fichier(chemin_meta)
tailles = pd.DataFrame([{"fichier": c.name, "brut Mio": round(c.stat().st_size / 2**20, 2),
                         "compressé gzip Mio": round(len(gzip.compress(c.read_bytes(), 9)) / 2**20, 2)}
                        for c in (chemin_bin, chemin_meta, chemin_entete)])
display(tailles)
print(f"modèle servi : {fichier_servi} ({DTYPE_NAVIGATEUR})")

,fichier,brut Mio,compressé gzip Mio
0,index_m2.f32,6.21,5.70
1,passages_m2.json,6.13,1.27
2,index_m2.json,0.00,0.00


modèle servi : onnx/model_quantized.onnx (q8)


**Pour la soutenance.**

- Un format simple et vérifiable : une matrice binaire, un tableau JSON aligné, un en-tête qui porte les empreintes.
- L'export est relu et comparé à la mémoire : ligne `i` de la matrice et passage `i` du JSON sont bien le même passage.
- L'en-tête dit au site quel modèle charger et comment encoder la question : préfixe, moyenne, normalisation, précision.

## 10. Export des index d'évaluation

Pour le notebook 03, chaque découpage est exporté avec la liste ordonnée de ses identifiants de passage. Les
métadonnées restent dans les tables du notebook 01, que l'on retrouve par ces identifiants.

In [14]:
manifeste_variantes = {}
for nom, E in INDEX.items():
    table, colonne = DECOUPAGES[nom]
    chemin_v = VARIANTES / f"{nom}.f32"
    E.astype("<f4").tofile(chemin_v)
    chemin_ids = VARIANTES / f"{nom}.ids.json"
    chemin_ids.write_text(json.dumps(TABLES[table].passage_id.tolist()), encoding="utf-8")
    assert np.array_equal(np.fromfile(chemin_v, dtype="<f4").reshape(-1, DIMENSION), E)
    manifeste_variantes[nom] = {"index": chemin_v.name, "identifiants": chemin_ids.name, "nb_vecteurs": int(E.shape[0]),
                                "table_notebook_01": manifeste01["methodes"][table]["fichier"], "colonne_encodee": colonne,
                                "sha256_index": sha256_fichier(chemin_v)}
(VARIANTES / "variantes.json").write_text(json.dumps(manifeste_variantes, ensure_ascii=False, indent=2), encoding="utf-8")
display(pd.DataFrame(manifeste_variantes).T[["nb_vecteurs", "colonne_encodee", "table_notebook_01"]])

,nb_vecteurs,colonne_encodee,table_notebook_01
m2,4240,entree_encodeur,data/decoupage/passages/passages_m2.parquet
m2_sans_entete,4240,entree_sans_entete,data/decoupage/passages/passages_m2.parquet
m2_256,7009,entree_encodeur,data/decoupage/passages/passages_m2_256.parquet
m0,5284,entree_encodeur,data/decoupage/passages/passages_m0.parquet
m1,5317,entree_encodeur,data/decoupage/passages/passages_m1.parquet
m3,4229,entree_encodeur,data/decoupage/passages/passages_m3.parquet


**Pour la soutenance.**

- Le notebook 03 n'aura rien à encoder côté passages : il comparera les découpages sur le jeu de questions, à partir de ces index.
- Les questions du jeu d'évaluation devront, elles, être encodées dans le navigateur, avec la précision servie.

## 11. Exemples de recherche

Le ticket #30 demande que plusieurs requêtes renvoient des résultats classés. Voici les quatre requêtes du support
pédagogique sur l'index du site. Ce sont des **illustrations** : le support en commente les réussites et l'échec
(la période d'essai d'un CDI), examiné juste après. Les requêtes sont encodées ici en fp32 ; la section 12 rejoue
des requêtes dans le navigateur, avec la précision servie.

In [15]:
Q_ex = encoder([PREFIXE_REQUETE + q for q in REQUETES_EXEMPLE], "ref", "exemples")
S_ex, top_ex = classement(Q_ex, E_site, k=3)
lignes = []
for i, q in enumerate(REQUETES_EXEMPLE):
    for rang, j in enumerate(top_ex[i], 1):
        r = P_site.iloc[j]
        lignes.append({"requête": q if rang == 1 else "", "rang": rang, "score": round(float(S_ex[i, j]), 4),
                       "fiche > section": f"{r.titre_fiche[:40]} > {r.section_titre[:45]}", "lien": r.url_citation[:70]})
display(pd.DataFrame(lignes))

,requête,rang,score,fiche > section,lien
0,Quelles mentions sont interdites sur le bulletin de paie ?,1,0.9022,Le bulletin de paie > Et les mentions interdites ?,https://travail-emploi.gouv.fr/le-bulletin-de-paie#:~:text=Aucune%20me
1,,2,0.8927,Le bulletin de paie > Chapô,https://travail-emploi.gouv.fr/le-bulletin-de-paie#:~:text=Au%20moment
2,,3,0.8839,Le bulletin de paie > Quelles sont les mentions obligatoires ?,https://travail-emploi.gouv.fr/le-bulletin-de-paie#:~:text=Les%20menti
3,Mon employeur peut-il me licencier pendant un arrêt maladie ?,1,0.8846,Les absences liées à la maladie ou à l'a > Peut-il y avoir licenciement pour maladie ?,https://travail-emploi.gouv.fr/les-absences-liees-la-maladie-ou-laccid
4,,2,0.8835,Les absences liées à la maladie ou à l'a > Chapô,https://travail-emploi.gouv.fr/les-absences-liees-la-maladie-ou-laccid
5,,3,0.8796,Les absences liées à la maladie ou à l'a > Quels sont les droits et obligations du salar,https://travail-emploi.gouv.fr/les-absences-liees-la-maladie-ou-laccid
6,Combien de temps peut durer la période d'essai d'un CDI ?,1,0.8927,Le contrat à durée déterminée (CDD) > Quelle est la durée de la période d'essai ?,https://travail-emploi.gouv.fr/le-contrat-duree-determinee-cdd#:~:text
7,,2,0.8873,CDD multi-remplacement : relance de l'ex > Terme du contrat,https://travail-emploi.gouv.fr/cdd-multi-remplacement-relance-de-lexpe
8,,3,0.8854,La période de reconversion > Que se passe-t-il au terme de la période d’es,https://travail-emploi.gouv.fr/la-periode-de-reconversion#:~:text=Si%2
9,Peut-on être viré quand on est en arrêt de travail ?,1,0.8764,Les absences liées à la maladie ou à l'a > Chapô,https://travail-emploi.gouv.fr/les-absences-liees-la-maladie-ou-laccid


**L'échec, examiné.** Pour la période d'essai d'un CDI, les trois premiers passages portent sur le CDD ou sur
d'autres contrats, alors que la fiche « La période d'essai » répond à la question, et plus précisément son passage
qui donne les durées maximales du CDI, selon l'article L. 1221-19. On cherche le rang de la fiche (son premier
passage) et celui de ce passage, avec le sigle, puis avec le terme développé. Une seule requête : c'est une piste pour le jeu
de questions du notebook 03 (y inclure des sigles), pas une mesure.

In [16]:
FICHE_ATTENDUE = "La période d'essai"
VARIANTES_CDI = ["Combien de temps peut durer la période d'essai d'un CDI ?",
                 "Combien de temps peut durer la période d'essai d'un contrat à durée indéterminée ?"]
Q_cdi = encoder([PREFIXE_REQUETE + q for q in VARIANTES_CDI], "ref", "exemple_cdi")
S_cdi, ordre_cdi = classement(Q_cdi, E_site, k=len(P_site))
dans_fiche = P_site.titre_fiche.to_numpy() == FICHE_ATTENDUE
lignes_attendues = np.flatnonzero(dans_fiche)
# trois passages de la fiche citent l'article ; celui qui donne les durées maximales du CDI commence par cette phrase
DEBUT_REPONSE = "Au terme de l’article L. 1221-19 du Code du travail, le contrat de travail à durée indéterminée"
passage_reponse = np.flatnonzero(dans_fiche & P_site.texte.str.startswith(DEBUT_REPONSE).to_numpy())
assert len(lignes_attendues) > 0 and len(passage_reponse) == 1, (len(lignes_attendues), len(passage_reponse))
for i, (q, ordre) in enumerate(zip(VARIANTES_CDI, ordre_cdi)):
    rang_fiche = 1 + int(np.flatnonzero(np.isin(ordre, lignes_attendues))[0])
    rang_passage = 1 + int(np.flatnonzero(ordre == passage_reponse[0])[0])
    ecart = S_cdi[i, ordre[0]] - S_cdi[i, passage_reponse[0]]
    print(f"{q}\n   rang de la fiche : {rang_fiche} | rang du passage L. 1221-19 : {rang_passage}, à {ecart:.4f} du premier "
          f"| premier résultat : {P_site.titre_fiche.iloc[ordre[0]][:45]}")

Combien de temps peut durer la période d'essai d'un CDI ?
   rang de la fiche : 4 | rang du passage L. 1221-19 : 5, à 0.0089 du premier | premier résultat : Le contrat à durée déterminée (CDD)
Combien de temps peut durer la période d'essai d'un contrat à durée indéterminée ?
   rang de la fiche : 1 | rang du passage L. 1221-19 : 2, à 0.0025 du premier | premier résultat : La période d'essai


## 12. Contrôle 4 : intégrité de l'export, vérifiée dans le navigateur

**Question.** Le site, qui lit l'en-tête, la matrice binaire et le JSON, classe-t-il les passages exactement comme
Python le ferait sur ces mêmes fichiers ? Ce contrôle vérifie l'ordre des octets, l'alignement du JSON sur la
matrice, le modèle et les options lus dans l'en-tête. Il ne dépend pas de la précision du modèle : pour chaque
requête, Python recalcule le classement **avec le vecteur produit par le navigateur**.

**Procédure.** Cette cellule écrit `controle/attendu.json` (24 requêtes). La page
`scripts/indexation/controle_navigateur.html`, servie en local depuis la racine du dépôt, lit l'export du site,
charge le modèle indiqué par l'en-tête, encode les requêtes et calcule les 5 premiers passages. Ses résultats sont
enregistrés dans `controle/resultats_navigateur.json`.

**Critères fixés à l'avance.** Le navigateur a chargé la précision de l'en-tête ; les empreintes SHA-256 qu'il
calcule sur la matrice et sur le JSON reçus sont celles de l'en-tête ; pour chaque requête, ses 5 premiers passages
sont ceux que Python calcule avec son vecteur, dans le même ordre, à des ex aequo près (scores séparés de moins
de 1e-6).

In [17]:
rng = np.random.default_rng(GRAINE + 1)
requetes_nav = REQUETES_EXEMPLE + [str(t) for t in rng.choice(TITRES_QUESTIONS, N_REQUETES_NAVIGATEUR, replace=False)]
attendu = {"requetes": [{"question": q} for q in requetes_nav], "entete_index": entete_index}
(CONTROLE / "attendu.json").write_text(json.dumps(attendu, ensure_ascii=False), encoding="utf-8")
print(f"attendu.json écrit : {len(requetes_nav)} requêtes")

resultats_nav = CONTROLE / "resultats_navigateur.json"
CONTROLE_NAVIGATEUR = None
if resultats_nav.exists():
    nav = json.loads(resultats_nav.read_text(encoding="utf-8"))
    assert [r["question"] for r in nav["resultats"]] == requetes_nav, "le navigateur a traité d'autres requêtes"
    assert nav.get("dtype") == DTYPE_NAVIGATEUR, f"le navigateur a chargé {nav.get('dtype')}, l'en-tête indique {DTYPE_NAVIGATEUR}"
    # empreintes calculées par la page sur les octets reçus : le navigateur a lu exactement les fichiers exportés
    assert nav.get("sha256_index") == entete_index["index"]["sha256"], "le navigateur a lu un autre index que celui exporté"
    assert nav.get("sha256_metadonnees") == entete_index["metadonnees"]["sha256"], "le navigateur a lu d'autres métadonnées"
    identiques, ecarts = 0, []
    for i, r in enumerate(nav["resultats"]):
        v = np.array(r["vecteur"], dtype=np.float32)
        s_py = E_site @ v
        attendu_top = np.argsort(-s_py, kind="stable")[:5]
        nav_top = [P_site.index[P_site.passage_id == pid][0] for pid in r["top5"]]
        # ex aequo : un écart d'ordre n'est accepté que si les scores concernés sont égaux à la tolérance près
        ok = all(a == b or abs(s_py[a] - s_py[b]) < TOLERANCE_EGALITE for a, b in zip(attendu_top, nav_top))
        identiques += ok
        ecarts.append(float(np.abs(np.array(r["scores5"]) - s_py[nav_top]).max()))
    cos_python_q8 = None
    if DTYPE_NAVIGATEUR == "q8":
        V_ctrl = np.array([r["vecteur"] for r in nav["resultats"]], dtype=np.float32)
        Q_ctrl = encoder([PREFIXE_REQUETE + q for q in requetes_nav], "q8", "navigateur")
        cos_python_q8 = float(np.einsum("ij,ij->i", V_ctrl, Q_ctrl).min())
    CONTROLE_NAVIGATEUR = {"requetes": len(requetes_nav), "précision chargée": nav["dtype"],
                           "top 5 identique à celui de Python": f"{identiques} / {len(requetes_nav)}",
                           "écart maximal entre scores du navigateur et de Python": f"{max(ecarts):.1e}",
                           "cosinus minimal avec le q8 de Python (information)": None if cos_python_q8 is None else round(cos_python_q8, 5),
                           "recherche exhaustive et tri, ms par requête (médiane)":
                               round(float(np.median([r["recherche_ms"] for r in nav["resultats"]])), 1),
                           "Transformers.js": nav.get("transformers_js"), "durée totale ms": round(nav.get("duree_ms", 0))}
    display(pd.Series(CONTROLE_NAVIGATEUR).to_frame("contrôle navigateur"))
    assert identiques == len(requetes_nav), "le navigateur ne classe pas les passages comme Python sur l'export"
else:
    print("contrôle d'intégrité non encore effectué : ouvrir la page de contrôle, puis réexécuter le notebook")

attendu.json écrit : 24 requêtes


,contrôle navigateur
requetes,24
précision chargée,q8
top 5 identique à celui de Python,24 / 24
écart maximal entre scores du navigateur et de Python,6.0e-08
cosinus minimal avec le q8 de Python (information),0.99419
"recherche exhaustive et tri, ms par requête (médiane)",1.7
Transformers.js,4.3.0
durée totale ms,1287


**Pour la soutenance.**

- Le dernier contrôle se fait dans un vrai navigateur, sur les fichiers exportés : c'est la chaîne que le site exécutera.
- Il isole l'intégrité de l'export (octets, alignement, en-tête) de la précision du modèle, mesurée en section 8.

## 13. Manifeste et synthèse

Le manifeste fixe tout ce qui permet de reproduire ou de contrôler l'indexation : entrées et leurs empreintes,
modèles et révisions, empreintes des fichiers ONNX, configuration retenue, résultats des contrôles, fichiers
produits. Deux exécutions doivent produire des index identiques octet pour octet.

In [18]:
manifeste02 = {
    "notebook": "scripts/indexation/02_indexation.ipynb", "execute_le": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "entrees": {code: manifeste01["methodes"][code] for code in TABLES},
    "modeles": {"reference": MODELE_REF, "onnx": {**MODELE_ONNX, "fichiers": FICHIERS_ONNX}},
    "controles": {
        "parite_cosinus_minimal": float(cos_parite.min()), "parite_seuil": SEUIL_PARITE,
        "quantification_python": tableau_quant.to_dict(orient="index"), "mcnemar_python": test, "configuration": CONFIGURATION,
        "fidelite_navigateur": None if FIDELITE is None else FIDELITE.to_dict(orient="index"),
        "precision_servie": DTYPE_NAVIGATEUR, "integrite_navigateur": CONTROLE_NAVIGATEUR,
    },
    "site": entete_index, "variantes": manifeste_variantes, "versions": VERSIONS,
    "durees_encodage_s": {f"{n}/{m}": round(d, 1) for (n, m), d in DUREES.items()},
}
(SORTIES / "manifeste.json").write_text(json.dumps(manifeste02, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
synthese = pd.Series({
    "passages indexés pour le site": f"{E_site.shape[0]} × {DIMENSION}",
    "parité ONNX fp32 et référence, cosinus minimal": f"{cos_parite.min():.7f}",
    "configuration retenue": f"{CONFIGURATION} ({'index fp32, requête q8' if CONFIGURATION == 'A' else 'index q8, requête q8'})",
    "précision servie au navigateur": DTYPE_NAVIGATEUR,
    "même premier résultat qu'en fp32 intégral, mesuré dans le navigateur":
        f"{FIDELITE.loc[DTYPE_NAVIGATEUR, 'même premier résultat %']} %" if FIDELITE is not None else "mesure à effectuer",
    "même premier résultat, simulation Python du q8": f"{tableau_quant.loc['A : index fp32, requête q8', 'même premier résultat %']} %",
    "intégrité de l'export dans le navigateur": CONTROLE_NAVIGATEUR["top 5 identique à celui de Python"] if CONTROLE_NAVIGATEUR else "à effectuer",
    "index d'évaluation exportés": ", ".join(INDEX),
})
display(synthese.to_frame("synthèse"))

,synthèse
passages indexés pour le site,4240 × 384
"parité ONNX fp32 et référence, cosinus minimal",0.9999998
configuration retenue,"A (index fp32, requête q8)"
précision servie au navigateur,q8
"même premier résultat qu'en fp32 intégral, mesuré dans le navigateur",91.8 %
"même premier résultat, simulation Python du q8",96.5 %
intégrité de l'export dans le navigateur,24 / 24
index d'évaluation exportés,"m2, m2_sans_entete, m2_256, m0, m1, m3"


## Ce que ce notebook établit, et ce qu'il n'établit pas

**Établi par les cellules ci-dessus** : les entrées sont celles du notebook 01 ; le modèle ONNX fp32 calcule les
mêmes vecteurs que la référence, en Python comme dans le navigateur ; l'effet de la quantification sur le
classement est mesuré dans le navigateur, qui fait foi, et la précision servie est choisie par une règle fixée à
l'avance ; l'export du site est relu à l'identique, et le navigateur classe les passages exactement comme Python
sur ces mêmes fichiers.

**Non établi** : la qualité de la recherche et la comparaison des découpages. Il faut pour cela le jeu de questions
annotées ; ce sera le notebook 03, qui partira des index de `data/indexation/variantes/`.